In [1]:
!sage -pip install pycryptodome
!sage -pip install pwn


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 3.1 MB/s  0:00:00 eta 0:00:01
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.9/12.9 MB 3.0 MB/s  0:00:04 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 2.4 MB/s  0:00:01 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 2.3 MB/s  0:00:02 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 2.5 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.3/16.3 MB 2.7 MB/s  0:00:05m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 2.9 MB/s  0:00:01 eta 0:00:01
  Created wheel for pwn: filename=pwn-1.0-py3-none-any.whl size=1247 sha256=631365b3dbc84aa4e08350e32a3ab1077455c523365a91cf88e4bb4c7ef6f8c6
  Stored in directory: /home/sage/.cache/pip/wheels/a2/56/8e/a9170a9dbc159fb2d0964640906cdd2fb6b08fb9f331cee112
Successfully

In [2]:
from Crypto.Random.random import getrandbits
import random
from utils import listener

SEED = getrandbits(32)
FLAG = b'crypto{????????????????????}'
q = 127


class Challenge():
    def __init__(self):
        self.before_input = f"Welcome to the LWE sample generator! Retrieve a sample using the 'get_sample' option, or reset the distribution using the 'reset' option.\n"
        self.rand = random.Random(SEED)

    def challenge(self, your_input):
        if not "option" in your_input:
            return {"error": "You must send an option to this server"}

        elif your_input["option"] == "reset":
            self.rand.seed(SEED)
            return {"success": "The distribution has been reset"}

        elif your_input["option"] == "get_sample":
            a = []
            for i in range(len(FLAG)):
                a.append(self.rand.randint(0, q - 1))

            e = self.rand.randint(-1, 1)

            self.rand.seed(getrandbits(32))
            if self.rand.randint(0, 1):
                a[self.rand.randint(0, len(a) - 1)] = self.rand.randint(0, q - 1)

            b = 0
            for (i, j) in zip(a, FLAG):
                b += i * j
            b += e
            b %= q

            return {"a": a, "b": b}

        else:
            return {"error": "Invalid option"}


import builtins; builtins.Challenge = Challenge # hack to enable challenge to be run locally, see https://cryptohack.org/faq/#listener
"""
When you connect, the 'challenge' function will be called on your JSON
input.
"""
listener.start_server(port=13390)


ImportError: cannot import name 'listener' from 'utils' (/home/sage/notebooks/utils.py)

In [74]:
q = 127
Zn = Integers(q)

from pwn import remote
import json

io = remote("socket.cryptohack.org", 13390)
io.recvline()  # message de bienvenue

def send(obj):
    io.sendline(json.dumps(obj).encode())
    return json.loads(io.recvline().decode())

def reset_serv():
    return send({"option": "reset"})

def sample_serv():
    return send({"option": "get_sample"})  # renvoie {"a": [...], "b": ...}



def first_sample():
    reset_serv()
    s = sample_serv()
    return vector(Zn, s["a"]), Zn(s["b"])

def find_row():
    a1, b1 = first_sample()
    a2, b2 = first_sample()
    while a1 == a2:
        a2, b2 = first_sample()
    return a1 - a2, b1 - b2

    


n = len(first_sample()[0])

M = matrix(Zn, 0, n)
vals = []
while M.rank() < n:
    row, val = find_row()
    candidate = M.stack(row)
    if candidate.rank() > M.rank():
        M = candidate
        vals.append(val)

B = vector(Zn, vals)
f = M.solve_right(B)

from Crypto.Util.number import isPrime, bytes_to_long, inverse, long_to_bytes
f1 = [str(long_to_bytes(int(x)).decode()) for x in f]
flag = ""
for x in f1:
    flag+=str(x)
print(flag)

[x] Opening connection to socket.cryptohack.org on port 13390
[x] Opening connection to socket.cryptohack.org on port 13390: Trying 134.122.111.232
[+] Opening connection to socket.cryptohack.org on port 13390: Done
crypto{f4ult_4ttack5_0n_lw3}
